In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import json
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from results_loader import METHODS, load_results

plt.rcParams.update({
    "text.usetex": True,
    "font.size": 20,
})

In [2]:
methods = list(METHODS)
records = load_results("../outputs/results", dataset="harvard", asr_model="whisper-tiny")  # aligned by sentence across methods
results = [dict() for _ in methods]

## Runtime & Budget

In [5]:
jsons = [[r["summary_path"] for r in records[m]] for m in methods]

In [6]:
# Runtime: end-to-end wall clock per sentence; budget: generations x population size
for m, r in zip(methods, results):
    times = [rec["elapsed_seconds"] for rec in records[m]]
    budget = [rec["budget"] for rec in records[m]]
    r |= {"runt": times, "budget": budget}
    print(f"{m} - Runtime: {np.mean(times):.2f} pm {np.std(times):.2f} s")
    print(f"{m} - Budget: {np.mean(budget):.2f} pm {np.std(budget):.2f}")

    ## Statistical Test

In [7]:
alternative = {
    "runt": "less",
    "budget": "less",
}

In [8]:
def cohens_d_paired(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    n = min(len(a), len(b))
    diff = a[:n] - b[:n]
    return abs(np.mean(diff) / np.std(diff, ddof=1))

In [9]:
for i, m in enumerate(results[1:], 1):
    print(f"{methods[0]} vs {methods[i]}")
    for k, vals in results[0].items():
        stat, p = wilcoxon(vals, m[k], alternative=alternative[k])
        d = cohens_d_paired(vals, m[k])
        print(f"\t{k}: p={p:.3e}; Cohens-D: {d:.1f}")

## Convergence Plot

In [12]:
fig, ax = plt.subplots(figsize=(8, 5))
num_generations = max(int(r["generations"]) for m in ["GATAS", "Waveform"] for r in records[m])
gens = np.arange(1, num_generations + 1)

# GATAS and Waveform report the generation at which they stopped (thresholds met or budget used)
for method in ["GATAS", "Waveform"]:
    curves = []
    for r in records[method]:
        # Early stopping is disabled: generation_found is the first generation meeting the thresholds
        found_at = r["generation_found"]
        if found_at is not None:
            cumul = (gens >= int(found_at)).astype(float)
        else:
            cumul = np.zeros_like(gens, dtype=float)
        curves.append(cumul)

    X = np.vstack(curves)
    mean = np.nanmean(X, axis=0)

    ax.plot(gens, mean, label=method)
    ax.fill_between(gens, 0, mean, alpha=0.2)

ax.set_xlabel("Generation")
ax.set_ylabel("Cumulative Success Rate")
ax.set_xlim(1, num_generations)
ax.set_ylim(0, 1)
ax.legend()
ax.legend(loc="upper left")
ax.grid(axis="y")

plt.savefig("_figures/convergance.pdf", dpi=200, bbox_inches="tight")

## Success vs. query budget

GATAS / Waveform: success of the archive after each generation (`archive_history.json`); baselines: one point at
their mean budget. PGD's budget counts white-box gradient iterations.

In [ ]:
import os
from results_loader import BUDGET_ABLATION, summary_table, success_at_budget

os.makedirs("_figures", exist_ok=True)
budgets = np.unique(np.logspace(1, 4, 30).astype(int))

def plot_budget_curves(records, ax):
    for m, recs in records.items():
        if not recs:
            continue
        if recs[0].get("archive_history"):
            ax.plot(budgets, [np.mean([success_at_budget(r, b) for r in recs]) for b in budgets], label=m)
        else:
            ax.scatter([np.mean([r["queries"] for r in recs])], [np.mean([r["success"] for r in recs])], label=m, zorder=3)
    ax.set_xscale("log")
    ax.set_xlabel("Attacked-ASR queries")
    ax.set_ylabel("Success rate")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, ds in zip(axes, ["harvard", "librispeech"]):
    plot_budget_curves(load_results("../outputs/results", dataset=ds, asr_model="whisper-tiny"), ax)
    ax.set_title(ds)
plt.savefig("_figures/success_vs_budget.pdf", bbox_inches="tight")

## Budget ablation (experimental)

GATAS with population 20 x 10 generations (main, 200 queries), 20 x 15 (300, SMACK's budget) and 100 x 100 (10,000).

In [ ]:
ablation = load_results("../outputs/results", dataset="harvard", asr_model="whisper-tiny", methods=BUDGET_ABLATION)
display(summary_table(ablation).round(3))

fig, ax = plt.subplots(figsize=(8, 5))
plot_budget_curves(ablation, ax)
plt.savefig("_figures/budget_ablation.pdf", bbox_inches="tight")